# Phenotype clustering
This notebook present an example of how the The Rapid's `optimize_stepMix()` class can be trained for class enumeration.
This code focuses on class enumeration, the key procedure and sometimes the major challenge in Latent Class Analysis (LCA), where the definition of the number of classes comprised in the sample happens.

In [1]:
import sys
sys.path.append('./code')
from AdjGridSearch7 import optimize_stepMix

In [2]:
import pandas as pd

In [3]:
import pickle
from stepmix.stepmix import StepMix
import time

# The dataset

The data used in this example comprise Dengue cases from Brazilian children 

- aged below 17 years old
- notified between 2017 and 2025
- notified within first 7 days from symptoms onset

- The data is alreay treated, and filtered for confirmed cases:
    - positivo: RESUL_PCR_==1 | RESUL_NS1==1 | RESUL_VI_N ==1 | IMUNOH_N==1  | HISTOPA_N ==1? além de SOROTIPO!=None
    - (DT_PCR - DT_SIN_PRI) ≤ 7 dias
    - (DT_NS1 - DT_SIN_PRI) ≤ 7 dias

In [4]:
data=pd.read_csv('./data/DenguePos_child_2017_2025.csv')

C:\Users\tomoe\AppData\Local\Temp\ipykernel_35444\2425457436.py:1: DtypeWarning: Columns (12,21,23,45,46,47,53,55,67,68,75,102,146,147) have mixed types. Specify dtype option on import or set low_memory=False.
  data=pd.read_csv('./data/DenguePos_child_2017_2025.csv')


In [5]:
data.head()

,Unnamed: 0,TP_NOT,ID_AGRAVO,DT_NOTIFIC,SEM_NOT,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_REGIONA,ID_UNIDADE,...,DENGUE_Inv,DENGUE_vir,Dengue_ntest,Chiks_ntest,ChiksConf,DT_DIGITA,MIGRADO_W,time_inv,Age_g,Year_block
0,1,2.0,A90,2017-08-05,201731,2017,12,120020,1941.0,6801099.0,...,1,1,1,0,NaN,NaN,NaN,0.0,9-12y,2017-2018
1,2,2.0,A90,2017-10-13,201741,2017,12,120020,1941.0,5336171.0,...,1,1,1,0,NaN,NaN,NaN,0.0,5-8y,2017-2018
2,3,2.0,A90,2017-03-09,201710,2017,12,120020,1941.0,6801099.0,...,1,1,1,0,NaN,NaN,NaN,3.0,13-16y,2017-2018
3,4,2.0,A90,2017-04-02,201714,2017,12,120040,1938.0,6439837.0,...,1,1,1,0,NaN,NaN,NaN,3.0,2-4y,2017-2018
4,5,2.0,A90,2017-12-22,201751,2017,12,120020,1941.0,6801099.0,...,1,1,2,0,NaN,NaN,NaN,0.0,<2y,2017-2018


In [6]:
data.columns

Index(['Unnamed: 0', 'TP_NOT', 'ID_AGRAVO', 'DT_NOTIFIC', 'SEM_NOT', 'NU_ANO',
       'SG_UF_NOT', 'ID_MUNICIP', 'ID_REGIONA', 'ID_UNIDADE',
       ...
       'DENGUE_Inv', 'DENGUE_vir', 'Dengue_ntest', 'Chiks_ntest', 'ChiksConf',
       'DT_DIGITA', 'MIGRADO_W', 'time_inv', 'Age_g', 'Year_block'],
      dtype='object', length=152)

# Variable list

The model (and phenotype) signification depends on the conjoint of the variables included. In our case, we use symptoms presented by the patients at the first seeking health assystance after symptoms onset

In [7]:
varList=['FEBRE', 'MIALGIA', 'CEFALEIA','EXANTEMA', 'VOMITO','NAUSEA','DOR_COSTAS','CONJUNTVIT',
         'ARTRITE','ARTRALGIA',#'ARTRITE_ARTRALGIA',#
         'PETEQUIA_N','LEUCOPENIA','LACO','DOR_RETRO']

All the variables are binary (presence vs absence of symptom), and they were measured just once, in a single point in time, making the Latent Class Analysis the adequate approach for phenotype clustering.

In [8]:
data[varList].head()

,FEBRE,MIALGIA,CEFALEIA,EXANTEMA,VOMITO,NAUSEA,DOR_COSTAS,CONJUNTVIT,ARTRITE,ARTRALGIA,PETEQUIA_N,LEUCOPENIA,LACO,DOR_RETRO
0,1,1,0,0,0,0,0,0,0,1,0,0,0,0
1,1,1,1,1,0,0,0,0,0,1,0,0,0,1
2,1,1,0,0,0,0,0,0,0,0,0,0,0,0
3,1,1,0,0,0,0,0,0,0,0,0,0,0,1
4,1,0,0,1,0,0,0,0,0,0,0,0,0,0


# Class enumeration - gridSearch 

The class enumeration, is the key procedure and sometimes the major challenge in Latent Class Analysis (LCA), where the definition of the number of classes comprised in the sample happens.
In the optimize_stepMix() class, this is done by conducting a grid search using the `.gridSearch()` method and providing a sequence of different k values (number of classes) to test. Usually this sequence starts in k=1.

We will test models from k=1 to k=15.
The `.gridSearch()` method display plots  related to fit metrics to enable the decision making.

In [ ]:

min_k=1
max_k=15
start_time = time.perf_counter()

gridS1_15Child=optimize_stepMix(data=data,
                 predictors=varList,outcome=None, random_state=42)
gridS1_15Child.gridSearch(low=min_k, high=max_k,  max_iter=2000,verbose=True, sample_weight=None
                   )
end_time = time.perf_counter()
execution_time = end_time - start_time

print(f"Execution time: {execution_time:.4f} seconds")

Testing 1 classes...
Testing 2 classes...
Testing 3 classes...


## let's store the object for future reference, and exploration

In [ ]:
with open('./res/childGridS/gs3_lca_1_15_max2000.pkl', 'wb') as file:
    pickle.dump(gridS1_15Child, file)

In [ ]:
import psutil
import platform

def report_system_config():
    print("--- System Configuration ---")
    print(f"OS: {platform.system()} {platform.release()}")
    print(f"Processor: {platform.processor()}")
    print(f"Physical Cores: {psutil.cpu_count(logical=False)}")
    print(f"Total RAM: {psutil.virtual_memory().total / (1024**3):.2f} GB")
    print("---------------------------")

def report_usage():
    print(f"CPU Usage: {psutil.cpu_percent()}%")
    print(f"RAM Usage: {psutil.virtual_memory().percent}%")

report_system_config()
report_usage()